# 03 · B3 — Does temporal context help, and how much is needed?
Window length L ∈ `L_grid` for all nine models (tabular models receive the flattened window), S-C protocol, multi-class task.

**H3:** temporal context (L > 1) improves minority-class recall over L = 1.

The best L is chosen on **validation** macro-F1 (averaged over the sequence models) and written to `config.json` for notebooks 04–06.

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 200, 'display.max_columns', 60)
from sdnids_bench.config import load_config, mode_params, active_datasets, enable_determinism
cfg = load_config('..')
mp = mode_params(cfg)
enable_determinism()
print('mode:', cfg['mode'], '| results:', cfg['out_dir'])
from sdnids_bench.experiments import run_grid, load_results, choose_best_L
from sdnids_bench.analysis import h3_summary
DATASETS = active_datasets(cfg, 'B3')
print('B3 on:', DATASETS, '| L grid:', mp['L_grid'], '| seeds:', mp['sweep_seeds'])

In [ ]:
run_grid(cfg, 'B3', DATASETS, ['multi'], ['SC'], mp['L_grid'], mp['sweep_seeds'], cfg['models'])

In [ ]:
res = load_results(cfg, DATASETS)
for ds in DATASETS:
    print(ds); t, s = h3_summary(res, ds); display(t.round(4)); print(s)
    display(res[(res.exp=='B3')&(res.dataset==ds)].pivot_table(index='model', columns='L', values='macro_f1', aggfunc='mean').round(4))

In [ ]:
cfg_disk = json.load(open('../config.json'))
best_L = choose_best_L({**cfg, 'best_L': None}, cfg['primary_dataset'])
print('best L (validation):', best_L)
cfg_disk['best_L'] = best_L
json.dump(cfg_disk, open('../config.json', 'w'), indent=2)